# D-MeZO — Bootstrap notebook for Google Colab Pro+

Target hardware: **RTX PRO 6000 Blackwell (96 GB)** или A100 80 GB.

Этот ноутбук:
1. Монтирует Google Drive (для чекпоинтов).
2. Клонирует или копирует проект `dmezo`.
3. Устанавливает зависимости.
4. Проверяет GPU.
5. Запускает Day 1 sanity check (MeZO на Qwen3-4B / SST-2).

**Бюджет:** ~30-50 compute units на полный прогон Day 1.

## 0. Mount Google Drive для чекпоинтов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS_DIR = '/content/drive/MyDrive/dmezo_runs'
os.makedirs(RUNS_DIR, exist_ok=True)
print(f'Runs will be saved to: {RUNS_DIR}')

## 1. Получить проект

Вариант A (если проект на GitHub):
```
!git clone https://github.com/<your>/dmezo.git /content/dmezo
```

Вариант B (если проект на Drive):
```
!cp -r /content/drive/MyDrive/dmezo /content/dmezo
```

Вариант C (быстрая загрузка через rsync с локальной машины Maksim — настроить отдельно).

In [ ]:
# Адаптируйте под свой способ доставки кода.
import os
if not os.path.exists('/content/dmezo'):
    !cp -r /content/drive/MyDrive/dmezo /content/dmezo
%cd /content/dmezo
!ls -la

## 2. Установить зависимости

In [ ]:
!pip install -q --upgrade pip
!pip install -q -e .
# Flash attention (опционально, ускоряет MeZO forward на Blackwell):
!pip install -q flash-attn --no-build-isolation || echo 'flash-attn install skipped'

## 3. Проверить GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    total_gb = torch.cuda.mem_get_info()[1] / 1e9
    print(f'GPU: {name}')
    print(f'Memory: {free_gb:.1f} / {total_gb:.1f} GB free')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')
    cc = torch.cuda.get_device_capability(0)
    print(f'Compute capability: {cc} (Blackwell = (10, 0) or higher)')

## 4. Запустить тесты

Должны пройти все: perturbation determinism + topology properties.

In [ ]:
!pytest tests/ -v

## 5. Day 1 sanity check — MeZO на Qwen3-4B / SST-2

Скрипт пишет JSONL-лог и чекпоинты в `experiments/01_sanity_qwen3_4b_sst2/` (внутри `/content/dmezo`). Чтобы сохранить в Drive, передадим `--output-dir`.

In [ ]:
OUTPUT_DIR = f'{RUNS_DIR}/01_sanity_qwen3_4b_sst2'
!python scripts/01_sanity_check_mezo.py --config configs/qwen3_4b_sst2.yaml --output-dir {OUTPUT_DIR}

## 6. Анализ результата

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

log_path = f'{OUTPUT_DIR}/log.jsonl'
rows = [json.loads(line) for line in open(log_path)]
df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train = df[df['train_loss'].notna()] if 'train_loss' in df.columns else df.iloc[0:0]
ev = df[df.get('eval_loss').notna()] if 'eval_loss' in df.columns else df.iloc[0:0]
if not train.empty:
    axes[0].plot(train['step'], train['train_loss'], label='train (moving avg)')
if not ev.empty:
    axes[0].plot(ev['step'], ev['eval_loss'], 'o-', label='eval')
axes[0].set_xlabel('MeZO step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Day 1 sanity: MeZO on Qwen3-4B / SST-2')
axes[0].legend()
axes[0].grid(alpha=0.3)

if 'projected_grad' in df.columns:
    pg = df[df['projected_grad'].notna()]
    axes[1].plot(pg['step'], pg['projected_grad'])
    axes[1].set_xlabel('MeZO step')
    axes[1].set_ylabel('Projected gradient')
    axes[1].set_title('Projected gradient (should be non-vanishing, non-NaN)')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sanity_plot.png', dpi=120)
plt.show()
print('Saved plot to', f'{OUTPUT_DIR}/sanity_plot.png')

## 7. Что дальше

Если sanity прошёл (eval loss упал на >10%):
- Переходить к Day 2 чтению FedKSeed/FedZeN.
- Запустить centralized baselines на BoolQ/COPA фоном.

Если sanity провалился:
- Проверить hyperparameters: `lr` в диапазоне 1e-7 - 1e-5, `eps=1e-3`.
- Проверить, что `param.requires_grad=True` для всех параметров.
- Распечатать `model.config` — убедиться, что Qwen3-4B загрузился именно как ожидалось.
- Запустить debug-версию с 10 steps и руками посмотреть на projected_grad.